# VaxiMère-QA-CG — Pipeline complet (Google Colab)

Construction du dataset d'intentions multilingue (FR / lingala / kituba) pour la
vaccination pédiatrique au Congo-Brazzaville.

**Étapes** : 1) install 2) vérif GPU 3) récupération du code 4) test rapide
(`dryrun`) 5) exécution complète 6) inspection des sorties.

In [ ]:
# 1) Installation des dépendances (Colab fournit déjà torch/transformers)
!pip install -q datasets transformers pandas accelerate sentencepiece huggingface_hub

In [ ]:
# 2) Vérification du GPU (un T4 suffit largement)
!nvidia-smi -L
import torch
print("CUDA disponible :", torch.cuda.is_available())
print("Device :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 3) Récupération du code
# Option A : cloner le dépôt GitHub (ajuster l'URL si besoin)
#!git clone https://github.com/maick-code/AIMS-Capstone.git
#%cd AIMS-Capstone

# Option B : si les fichiers sont déjà dans Colab (Upload), décommenter :
import os
print("Fichiers présents :", [f for f in os.listdir('.') if f.endswith('.py') or f.endswith('.md')])

In [ ]:
# 4) Test rapide SANS modèle ni réseau (valide tout le câblage)
!python run_pipeline.py --mode dryrun

In [ ]:
# 5) Exécution COMPLÈTE (télécharge mDeBERTa + NLLB-600M, exécute le pipeline)
#    Durée estimée sur T4 : ~10-20 min selon la connexion.
!python run_pipeline.py --mode full

In [ ]:
# 6) Inspection des sorties
import json
from pathlib import Path

data_dir = Path("data/final")
for f in sorted(data_dir.glob("*.jsonl")):
    n = sum(1 for _ in f.open(encoding="utf-8"))
    print(f"{f.name}: {n} lignes")

# Aperçu du JSONL principal
with open(data_dir / "vaximere_qa_cg_train.jsonl", encoding="utf-8") as fh:
    for line in fh:
        print(line.strip())
        break

# Statistiques
print(json.dumps(json.load(open(data_dir / "stats_report.json", encoding="utf-8")),
                 ensure_ascii=False, indent=2))

In [ ]:
# 7) Téléchargement des livrables
from google.colab import files
for name in ["vaximere_qa_cg_train.jsonl", "faq_validee.json", "DATA_CARD.md"]:
    p = Path("data/final") / name
    if p.exists():
        files.download(str(p))
    else:
        print(f"(absent) {name}")